# Data Preparation for Whisper Fine-tuning
Converts custom audio datasets (train & test) from Google Drive into the HuggingFace dataset format required by the Whisper fine-tuning pipeline.

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Install Dependencies

In [ ]:
!pip install datasets soundfile librosa

## Step 3: Configure Paths

In [ ]:
import os
from pathlib import Path

# Google Drive folder IDs
# Train: https://drive.google.com/drive/folders/1Zqk0-XClL5uKRpkGTH2F7EURz1IhmnZE
# Test:  https://drive.google.com/drive/folders/1DdgxjQGk1A6FuPNdQV5PcAmKtTj13DCE

TRAIN_DATA_DIR = Path("/content/drive/MyDrive")  # UPDATE: full path to train folder
TEST_DATA_DIR  = Path("/content/drive/MyDrive")  # UPDATE: full path to test folder

WORK_DIR       = Path("/content/whisper_data")
TRAIN_PREP_DIR = WORK_DIR / "train_source"
TEST_PREP_DIR  = WORK_DIR / "test_source"
TRAIN_OUT_DIR  = WORK_DIR / "train_dataset"
TEST_OUT_DIR   = WORK_DIR / "test_dataset"

for d in [TRAIN_PREP_DIR, TEST_PREP_DIR, TRAIN_OUT_DIR, TEST_OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Directories created:")
for d in [TRAIN_PREP_DIR, TEST_PREP_DIR, TRAIN_OUT_DIR, TEST_OUT_DIR]:
    print(f"  {d}")

## Step 4: Explore Data Structure

In [ ]:
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}

def explore_folder(folder: Path, label: str):
    files = list(folder.iterdir())
    audio_files = [f for f in files if f.suffix.lower() in AUDIO_EXTENSIONS]
    text_files  = [f for f in files if f.suffix.lower() == '.txt']
    print(f"\n{label}: {folder}")
    print(f"  Total files : {len(files)}")
    print(f"  Audio files : {len(audio_files)} (e.g. {audio_files[0].name if audio_files else 'none'})")
    print(f"  Text files  : {len(text_files)} (e.g. {text_files[0].name if text_files else 'none'})")
    return audio_files, text_files

train_audio, train_text = explore_folder(TRAIN_DATA_DIR, "TRAIN")
test_audio,  test_text  = explore_folder(TEST_DATA_DIR,  "TEST")

## Step 5: Build `audio_paths` and `text` Files

Each line format:
```
<unique-id> <absolute-path-to-audio>
<unique-id> <transcription>
```

> Assumes each audio file has a paired `.txt` file with the same stem containing the transcription.

In [ ]:
def build_manifest_files(audio_files, source_dir: Path, prep_dir: Path, split: str):
    """
    Creates audio_paths and text manifest files for data_prep.py.
    Expects each .wav to have a paired .txt file with the transcription.
    """
    audio_lines = []
    text_lines  = []
    missing     = []

    for idx, audio_file in enumerate(sorted(audio_files), start=1):
        txt_file = source_dir / (audio_file.stem + ".txt")
        if not txt_file.exists():
            missing.append(audio_file.name)
            continue

        utt_id = f"{split}_{idx:05d}"
        transcript = txt_file.read_text(encoding='utf-8').strip()

        audio_lines.append(f"{utt_id} {audio_file.resolve()}")
        text_lines.append(f"{utt_id} {transcript}")

    (prep_dir / "audio_paths").write_text("\n".join(audio_lines), encoding='utf-8')
    (prep_dir / "text").write_text("\n".join(text_lines), encoding='utf-8')

    print(f"[{split}] Wrote {len(audio_lines)} entries to {prep_dir}")
    if missing:
        print(f"  WARNING: {len(missing)} audio files had no matching .txt — skipped: {missing[:5]}")

build_manifest_files(train_audio, TRAIN_DATA_DIR, TRAIN_PREP_DIR, split="train")
build_manifest_files(test_audio,  TEST_DATA_DIR,  TEST_PREP_DIR,  split="test")

## Step 6: Verify Manifest Files

In [ ]:
def verify_manifests(prep_dir: Path, label: str, n: int = 3):
    audio_lines = (prep_dir / "audio_paths").read_text(encoding='utf-8').strip().splitlines()
    text_lines  = (prep_dir / "text").read_text(encoding='utf-8').strip().splitlines()

    print(f"\n{label}")
    print(f"  audio_paths entries : {len(audio_lines)}")
    print(f"  text entries        : {len(text_lines)}")
    print(f"  Counts match        : {len(audio_lines) == len(text_lines)}")
    print(f"\n  First {n} audio_paths entries:")
    for l in audio_lines[:n]: print(f"    {l}")
    print(f"\n  First {n} text entries:")
    for l in text_lines[:n]:  print(f"    {l}")

verify_manifests(TRAIN_PREP_DIR, "TRAIN")
verify_manifests(TEST_PREP_DIR,  "TEST")

## Step 7: Run Data Preparation (Convert to HuggingFace Dataset Format)

In [ ]:
from datasets import Dataset, Audio, Value

def run_data_prep(prep_dir: Path, output_dir: Path, label: str):
    scp_entries = (prep_dir / "audio_paths").read_text(encoding='utf-8').strip().splitlines()
    txt_entries = (prep_dir / "text").read_text(encoding='utf-8').strip().splitlines()

    if len(scp_entries) != len(txt_entries):
        raise ValueError(
            f"[{label}] Mismatch: {len(scp_entries)} audio entries vs {len(txt_entries)} text entries. "
            "Check your audio_paths and text files."
        )

    audio_paths   = [line.split(maxsplit=1)[1].strip() for line in scp_entries]
    transcriptions = [' '.join(line.split()[1:]).strip() for line in txt_entries]

    dataset = Dataset.from_dict({"audio": audio_paths, "sentence": transcriptions})
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))
    dataset = dataset.cast_column("sentence", Value("string"))
    dataset.save_to_disk(str(output_dir))

    print(f"[{label}] Data preparation complete → {output_dir}")
    print(f"  Samples : {len(dataset)}")
    print(f"  Features: {dataset.features}")
    return dataset

train_dataset = run_data_prep(TRAIN_PREP_DIR, TRAIN_OUT_DIR, "TRAIN")
test_dataset  = run_data_prep(TEST_PREP_DIR,  TEST_OUT_DIR,  "TEST")

## Step 8: Sanity Check — Inspect a Sample

In [ ]:
import IPython.display as ipd
import numpy as np

print("=== Train sample ===")
sample = train_dataset[0]
print(f"  Transcription : {sample['sentence']}")
print(f"  Sampling rate : {sample['audio']['sampling_rate']} Hz")
print(f"  Audio length  : {len(sample['audio']['array']) / sample['audio']['sampling_rate']:.2f} seconds")
ipd.display(ipd.Audio(sample['audio']['array'], rate=sample['audio']['sampling_rate']))

## Step 9: Summary

In [ ]:
print("Data Preparation Complete")
print(f"  Train dataset : {TRAIN_OUT_DIR}  ({len(train_dataset)} samples)")
print(f"  Test dataset  : {TEST_OUT_DIR}   ({len(test_dataset)} samples)")
print()
print("These paths can be passed directly to the fine-tuning scripts:")
print(f"  --train_datasets {TRAIN_OUT_DIR}")
print(f"  --eval_datasets  {TEST_OUT_DIR}")